In [ ]:
!pip install transformers datasets tokenizers

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained("textattack/bert-base-uncased-imdb")
tokenizer = AutoTokenizer.from_pretrained("textattack/bert-base-uncased-imdb")
# model_name = 'bert-base-uncased'
# tokenizer = BertTokenizer.from_pretrained(model_name)
# model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(DEVICE)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
from datasets import load_dataset

# Load the IMDB dataset (train or test split)
dataset = load_dataset("imdb", split="test")  # or "train"

# Extract texts and labels
texts = dataset["text"]
labels = dataset["label"]

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
# Custom Dataset
class MyDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len, return_tensors='pt')
        self.labels = labels

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

    def __len__(self):
        return len(self.labels)

# Classifier model on top of BERT
# class BERTClassifier(nn.Module):
#     def __init__(self, bert, num_classes):
#         super(BERTClassifier, self).__init__()
#         self.bert = bert
#         self.dropout = nn.Dropout(0.3)
#         self.classifier = nn.Linear(bert.config.hidden_size, num_classes)

#     def forward(self, input_ids, attention_mask):
#         outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
#         cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token
#         cls_output = self.dropout(cls_output)
#         logits = self.classifier(cls_output)
#         return logits

test_dataset = MyDataset(texts, labels, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=8)

In [ ]:
model.eval()
predictions, true_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        # print(logits)
        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())
print(classification_report(true_labels, predictions))

In [ ]:
# unique_labels = list(label_to_index.keys())

# print(classification_report(true_labels, predictions, target_names=unique_labels))


              precision    recall  f1-score   support

           0       0.89      0.89      0.89     12500
           1       0.89      0.90      0.89     12500

    accuracy                           0.89     25000
   macro avg       0.89      0.89      0.89     25000
weighted avg       0.89      0.89      0.89     25000



In [ ]:
from transformers import AutoTokenizer
import transformers
import torch
model = "TinyLlama/TinyLlama_v1.1"
tokenizer = AutoTokenizer.from_pretrained(model)
pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    torch_dtype=torch.float16,
    device_map="auto",
)

Device set to use cuda:0


In [ ]:
messages = [
    {"role": "System", "content": "You are a calculator who answers in as few characters as possible."},
    {"role": "User", "content": "What is 2+2?"},
    {"role": "Assistant", "content": "2+2=4"},
    {"role": "User", "content": "What is 6+2?"}
]
pythonsequences = pipeline(
    messages,
    do_sample=False,
    temperature=0.1,
    max_new_tokens=20,
    repetition_penalty=1.5,
    eos_token_id=tokenizer.eos_token_id,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")

ValueError: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templates and setting the tokenizer.chat_template attribute, please see the documentation at https://huggingface.co/docs/transformers/main/en/chat_templating

In [ ]:
pythonsequences = pipeline(
    messages,
    do_sample=False,
    temperature=0.1,
    max_new_tokens=20,
    repetition_penalty=1.5,
    eos_token_id=tokenizer.eos_token_id,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")

TypeError: unhashable type: 'list'

In [2]:
import torch
from transformers import pipeline

pipe = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", torch_dtype=torch.bfloat16, device_map="auto")


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
# messages = [
#     {
#         "role": "system",
#         "content": "You are a calculator",
#     },
#     {"role": "user", "content": "What is 2+233?"},
# ]
# prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
# outputs = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.3, top_k=50, top_p=0.95)
# print(outputs[0]["generated_text"])

<|system|>
You are a calculator</s>
<|user|>
What is 2+233?</s>
<|assistant|>
2 + 233 = 236


In [ ]:
input=[
{
"role": "system",
"content": "You are a highly accurate logical fallacy detector. When you see a text, you return the most prominent logical fallacy. If no logical fallacy is prominent, return \"none\"."
},
{
"role": "user",
"content": "Senator Randall isn't lying when she says she cares about her constituents—she wouldn't lie to people she cares about."
},
{
"role": "assistant",
"content": "Circular Reasoning"
},
{
"role": "user",
"content": "I love eating burgers."
},
{
"role": "assistant",
"content": "none"
},
{"role:" "user",
 "content:" "If we ban Hummers because they are bad for the environment, eventually the government will ban all cars, so we should not ban Hummers."}
]
prompt = pipe.tokenizer.apply_chat_template(input, tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.3, top_k=50, top_p=0.95)
print(outputs[0]["generated_text"])

In [ ]:
text1 = "Senator Randall isn't lying when she says she cares about her constituents—she wouldn't lie to people she cares about."
fallacy1 = "Circular Reasoning"

text2 = "I love eating burgers."
fallacy2 = "none"

text3 = "If we ban Hummers because they are bad for the environment, eventually the government will ban all cars, so we should not ban Hummers."
fallacy3 = "False Dilemma"

test = "You oppose a senator's proposal to extend government-funded health care to poor minority children because that senator is a liberal Democrat."

# Format prompt with variables using f-strings
prompt = f"""You are a logic expert. Identify the logical fallacy in each statement. If there is no fallacy, reply with "none".

Text: {text1}
Fallacy: {fallacy1}

Text: {text2}
Fallacy: {fallacy2}

Text: {text3}
Fallacy: {fallacy3}

Text: {test}
Fallacy:"""

print(prompt)

You are a logic expert. Identify the logical fallacy in each statement. If there is no fallacy, reply with "none".

Text: Senator Randall isn't lying when she says she cares about her constituents—she wouldn't lie to people she cares about.
Fallacy: Circular Reasoning

Text: I love eating burgers.
Fallacy: none

Text: If we ban Hummers because they are bad for the environment, eventually the government will ban all cars, so we should not ban Hummers.
Fallacy: False Dilemma

Text: You oppose a senator's proposal to extend government-funded health care to poor minority children because that senator is a liberal Democrat.
Fallacy:


In [ ]:
outputs = pipe(prompt, max_new_tokens=10, do_sample=True, temperature=0.1, top_k=20, top_p=0.8)
llm_output = outputs[0]["generated_text"]
print(llm_output)


You are a logic expert. Identify the logical fallacy in each statement. If there is no fallacy, reply with "none".

Text: Senator Randall isn't lying when she says she cares about her constituents—she wouldn't lie to people she cares about.
Fallacy: Circular Reasoning

Text: I love eating burgers.
Fallacy: none

Text: If we ban Hummers because they are bad for the environment, eventually the government will ban all cars, so we should not ban Hummers.
Fallacy: False Dilemma

Text: You oppose a senator's proposal to extend government-funded health care to poor minority children because that senator is a liberal Democrat.
Fallacy: False Equivalence

Text: The government


In [ ]:
import re
blocks = re.split(r'\n(?=Text:)', llm_output)

# Extract (text, fallacy) from each block
pairs = {}
for block in blocks:
    text_match = re.search(r'Text:\s*(.+)', block)
    fallacy_match = re.search(r'Fallacy:\s*(.+)', block)
    if text_match and fallacy_match:
        text = text_match.group(1).strip()
        fallacy = fallacy_match.group(1).strip()
        pairs[text]= fallacy

# Print result
print(pairs[test])

False Equivalence


In [ ]:
import re

# Dataset of texts and their corresponding fallacies
data = [
    ("Senator Randall isn't lying when she says she cares about her constituents—she wouldn't lie to people she cares about.", "Circular Reasoning"),
    ("I love eating burgers.", "none"),
    ("If we ban Hummers because they are bad for the environment, eventually the government will ban all cars, so we should not ban Hummers.", "False Dilemma"),
]

# The test text that you want to evaluate
test_text = "You oppose a senator's proposal to extend government-funded health care to poor minority children because that senator is a liberal Democrat."

# Generate the prompt dynamically from the dataset
prompt = "You are a logic expert. Identify the logical fallacy in each statement. If there is no fallacy, reply with 'none'.\n\n"
for text, fallacy in data:
    prompt += f"Text: {text}\nFallacy: {fallacy}\n"

# Add the test text to the prompt for analysis
prompt += f"Text: {test_text}\nFallacy:"

# Run the model (assuming `pipe` is your function to get outputs from the model)
outputs = pipe(prompt, max_new_tokens=10, do_sample=True, temperature=0.1, top_k=20, top_p=0.8)

# Get the model's output
llm_output = outputs[0]["generated_text"]

# Split output into blocks and extract (text, fallacy) pairs
blocks = re.split(r'\n(?=Text:)', llm_output)

# Create a dictionary to store the pairs
pairs = {}
for block in blocks:
    text_match = re.search(r'Text:\s*(.+)', block)
    fallacy_match = re.search(r'Fallacy:\s*(.+)', block)
    if text_match and fallacy_match:
        text = text_match.group(1).strip()
        fallacy = fallacy_match.group(1).strip()
        pairs[text] = fallacy

# Print the result for the test text
print(pairs.get(test_text, "Fallacy not found"))


False Equivalence


In [4]:
import re

# Training examples
data = [
    ("Senator Randall isn't lying when she says she cares about her constituents—she wouldn't lie to people she cares about.", "Circular Reasoning"),
    ("I love eating burgers.", "none"),
    ("If we ban Hummers because they are bad for the environment, eventually the government will ban all cars, so we should not ban Hummers.", "False Dilemma"),
]

# Array of test texts
test_texts = [
    "There is nothing to do.",
    "You oppose a senator's proposal to extend government-funded health care to poor minority children because that senator is a liberal Democrat.",
    "Aliens exist because nobody has proven they don't."
]

# Build base prompt with known examples
prompt = "You are a logic expert. Identify the logical fallacy in each statement. If there is no fallacy, reply with \"none\".\n\n"
# for text, fallacy in data:
#     prompt += f"Text: {text}\nFallacy: {fallacy}\n\n"
output = []
for test in test_texts:
    # prompt = "You are a logic expert. Identify the logical fallacy in each statement. If there is no fallacy, reply with \"none\".\n\n"
    for text, fallacy in data:
        prompt += f"Text: {text}\nFallacy: {fallacy}\n\n"
    prompt += f"Text: {test}\nFallacy:"
    # print(prompt)

    outputs = pipe(prompt, max_new_tokens=10, do_sample=True, temperature=0.1, top_k=20, top_p=0.8)
    llm_output = outputs[0]["generated_text"]

    fallacy_match = re.search(r'Fallacy:\s*(.*)', llm_output)
    fallacy = fallacy_match.group(1).strip() if fallacy_match else "Not found"

    # print(f"Text: {test}\nPredicted Fallacy: {fallacy}\n")
    blocks = re.split(r'\n(?=Text:)', llm_output)

    # Create a dictionary to store the pairs
    pairs = {}
    for block in blocks:
        text_match = re.search(r'Text:\s*(.+)', block)
        fallacy_match = re.search(r'Fallacy:\s*(.+)', block)
        if text_match and fallacy_match:
            text = text_match.group(1).strip()
            fallacy = fallacy_match.group(1).strip()
            pairs[text] = fallacy
    output.append(pairs.get(test, "Fallacy not found"))
    prompt = "You are a logic expert. Identify the logical fallacy in each statement. If there is no fallacy, reply with \"none\".\n\n"

print(output)

['none', 'False Equivalence', 'False Dilemma']


In [39]:
def generate_prompt():
    prompt = """You are a logic expert. For each statement, choose exactly one logical fallacy from the list below that best matches the reasoning error. If there is no fallacy, reply with "none".

Choose from:
- False Causality
- Fallacy of Logic
- Circular Reasoning
- Equivocation
- Faulty Generalization
- Fallacy of Relevance
- False Dilemma
- Ad Populum
- Fallacy of Extension
- Fallacy of Credibility
- Appeal to Emotion
- Intentional Fallacy
- Ad Hominem
- none

Examples:
"""
    return prompt


In [8]:
import re

# Training examples
data = [
    ("Senator Randall isn't lying when she says she cares about her constituents—she wouldn't lie to people she cares about.", "Circular Reasoning"),
    ("I love eating burgers.", "none"),
    ("If we ban Hummers because they are bad for the environment, eventually the government will ban all cars, so we should not ban Hummers.", "False Dilemma"),
]

# Array of test texts
test_texts = [
    "There is nothing to do.",
    "You oppose a senator's proposal to extend government-funded health care to poor minority children because that senator is a liberal Democrat.",
    "Aliens exist because nobody has proven they don't."
]

# Build base prompt with known examples
prompt = generate_prompt()
# for text, fallacy in data:
#     prompt += f"Text: {text}\nFallacy: {fallacy}\n\n"
output = []
for test in test_texts:
    # prompt = "You are a logic expert. Identify the logical fallacy in each statement. If there is no fallacy, reply with \"none\".\n\n"
    for text, fallacy in data:
        prompt += f"Text: {text}\nFallacy: {fallacy}\n\n"
    prompt += f"Text: {test}\nFallacy:"
    # print(prompt)

    outputs = pipe(prompt, max_new_tokens=10, do_sample=True, temperature=0.1, top_k=20, top_p=0.8)
    llm_output = outputs[0]["generated_text"]

    fallacy_match = re.search(r'Fallacy:\s*(.*)', llm_output)
    fallacy = fallacy_match.group(1).strip() if fallacy_match else "Not found"

    # print(f"Text: {test}\nPredicted Fallacy: {fallacy}\n")
    blocks = re.split(r'\n(?=Text:)', llm_output)

    # Create a dictionary to store the pairs
    pairs = {}
    for block in blocks:
        text_match = re.search(r'Text:\s*(.+)', block)
        fallacy_match = re.search(r'Fallacy:\s*(.+)', block)
        if text_match and fallacy_match:
            text = text_match.group(1).strip()
            fallacy = fallacy_match.group(1).strip()
            pairs[text] = fallacy
    output.append(pairs.get(test, "Fallacy not found"))
    prompt = generate_prompt()

print(output)

['none', 'Ad Populum', 'Fallacy of Extension']


In [16]:
import re

data = [
    ("Senator Randall isn't lying when she says she cares about her constituents—she wouldn't lie to people she cares about.", "Circular Reasoning"),
    ("I love eating burgers.", "none"),
    ("If we ban Hummers because they are bad for the environment, eventually the government will ban all cars, so we should not ban Hummers.", "False Dilemma"),
]

test_texts = [
    "There is nothing to do.",
    "You oppose a senator's proposal to extend government-funded health care to poor minority children because that senator is a liberal Democrat.",
    "Aliens exist because nobody has proven they don't."
]

def build_training_prompt(data):
    prompt = generate_prompt()
    for text, fallacy in data:
        prompt += f"Text: {text}\nFallacy: {fallacy}\n\n"
    return prompt

base_prompt = build_training_prompt(data)
output = []

for test in test_texts:
    full_prompt = f"{base_prompt}Text: {test}\nFallacy:"
    outputs = pipe(full_prompt, max_new_tokens=5, do_sample=True, temperature=0.1, top_k=20, top_p=0.8)
    llm_output = outputs[0]["generated_text"]
    pairs = {}
    blocks = re.split(r'\n(?=Text:)', llm_output)
    for block in blocks:
        text_match = re.search(r'Text:\s*(.+)', block)
        fallacy_match = re.search(r'Fallacy:\s*(.+)', block)
        if text_match and fallacy_match:
            text = text_match.group(1).strip()
            fallacy = fallacy_match.group(1).strip()
            pairs[text] = fallacy
    output.append(pairs.get(test, "Fallacy not found"))

print(output)


['none', 'Ad Populum', 'none']


In [18]:
!pip install transformers datasets tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which

In [23]:
from datasets import load_dataset, concatenate_datasets

ds = load_dataset("tasksource/logical-fallacy")

# Concatenate 'dev' and 'test' splits
dataSet = concatenate_datasets([ds['dev'], ds['test']])
texts = dataSet['source_article']
labels = dataSet['logical_fallacies']

In [36]:
from tqdm import tqdm
def build_training_prompt(data):
    prompt = generate_prompt()
    for text, fallacy in data:
        prompt += f"Text: {text}\nFallacy: {fallacy}\n\n"
    return prompt

base_prompt = build_training_prompt(data)
output = []

for test in tqdm(texts, desc="Classifying fallacies"):
    full_prompt = f"{base_prompt}Text: {test}\nFallacy:"
    outputs = pipe(full_prompt, max_new_tokens=5, do_sample=True, temperature=0.1, top_k=20, top_p=0.8)
    llm_output = outputs[0]["generated_text"]
    pairs = {}
    blocks = re.split(r'\n(?=Text:)', llm_output)
    for block in blocks:
        text_match = re.search(r'Text:\s*(.+)', block)
        fallacy_match = re.search(r'Fallacy:\s*(.+)', block)
        if text_match and fallacy_match:
            text = text_match.group(1).strip()
            fallacy = fallacy_match.group(1).strip()
            pairs[text] = fallacy
    output.append(pairs.get(test, "Fallacy not found"))

Classifying fallacies: 100%|██████████| 1081/1081 [02:31<00:00,  7.13it/s]


In [37]:
from sklearn.metrics import classification_report
output = [label.lower() for label in output]
# Print classification report with text labels
print(classification_report(labels, output))


                        precision    recall  f1-score   support

            ad hominem       0.00      0.00      0.00       109
            ad populum       0.13      0.44      0.20        86
     appeal to emotion       0.00      0.00      0.00        91
    circular reasoning       0.00      0.00      0.00        38
          equivocation       0.00      0.00      0.00        18
     fallacy not found       0.00      0.00      0.00         0
fallacy of credibility       0.00      0.00      0.00        63
        fallacy of ext       0.00      0.00      0.00         0
  fallacy of extension       0.00      0.00      0.00        66
        fallacy of log       0.00      0.00      0.00         0
      fallacy of logic       0.00      0.00      0.00        64
         fallacy of re       0.00      0.00      0.00         0
  fallacy of relevance       0.00      0.00      0.00        90
       false causality       0.07      0.04      0.05        77
         false dilemma       0.00      

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_

In [40]:
print(build_training_prompt(data))

You are a logic expert. For each statement, choose exactly one logical fallacy from the list below that best matches the reasoning error. If there is no fallacy, reply with "none".

Choose from:
- False Causality
- Fallacy of Logic
- Circular Reasoning
- Equivocation
- Faulty Generalization
- Fallacy of Relevance
- False Dilemma
- Ad Populum
- Fallacy of Extension
- Fallacy of Credibility
- Appeal to Emotion
- Intentional Fallacy
- Ad Hominem
- none

Examples:
Text: Senator Randall isn't lying when she says she cares about her constituents—she wouldn't lie to people she cares about.
Fallacy: Circular Reasoning

Text: I love eating burgers.
Fallacy: none

Text: If we ban Hummers because they are bad for the environment, eventually the government will ban all cars, so we should not ban Hummers.
Fallacy: False Dilemma




In [2]:
from google.colab import userdata
key = userdata.get('OPENAI_API_KEY')
from openai import OpenAI
client = OpenAI(api_key=key)

response = client.responses.create(
  model="gpt-4.1-mini",
  input=[
    {
      "role": "system",
      "content": [
        {
          "type": "input_text",
          "text": "You are a logic expert. For each statement, choose exactly one logical fallacy from the list below that best matches the reasoning error. If there is no fallacy, reply with \"none\".\n\nChoose from:\n- False Causality\n- Fallacy of Logic\n- Circular Reasoning\n- Equivocation\n- Faulty Generalization\n- Fallacy of Relevance\n- False Dilemma\n- Ad Populum\n- Fallacy of Extension\n- Fallacy of Credibility\n- Appeal to Emotion\n- Intentional Fallacy\n- Ad Hominem\n- none"
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "input_text",
          "text": "Senator Randall isn't lying when she says she cares about her constituents—she wouldn't lie to people she cares about."
        }
      ]
    },
    {
      "role": "assistant",
      "content": [
        {
          "type": "output_text",
          "text": "Circular Reasoning"
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "input_text",
          "text": "I love eating burgers"
        }
      ]
    },
    {
      "role": "assistant",
      "content": [
        {
          "type": "output_text",
          "text": "none"
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "input_text",
          "text": "If we ban Hummers because they are bad for the environment, eventually the government will ban all cars, so we should not ban Hummers."
        }
      ]
    },
    {
      "role": "assistant",
      "content": [
        {
          "type": "output_text",
          "text": "False Dilemma"
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "input_text",
          "text": "I know the Professor said that the Bridges of Madison County was smarmy trash and lacked any artistic worth. But I still think he's wrong. After all, it was on the best-seller list for over 100 weeks."
        }
      ]
    }
  ],
  text={
    "format": {
      "type": "text"
    }
  },
  reasoning={},
  tools=[],
  temperature=0.1,
  max_output_tokens=20,
  top_p=0.8,
  store=True
)
print(response.output_text)

Fallacy of Credibility


In [9]:
def get_fallacy_response(client, text_comment):
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=[
            {
                "role": "system",
                "content": [
                    {
                        "type": "input_text",
                        "text": "You are a logic expert. For each statement, choose exactly one logical fallacy from the list below that best matches the reasoning error. If there is no fallacy, reply with \"none\".\n\nChoose from:\n- False Causality\n- Fallacy of Logic\n- Circular Reasoning\n- Equivocation\n- Faulty Generalization\n- Fallacy of Relevance\n- False Dilemma\n- Ad Populum\n- Fallacy of Extension\n- Fallacy of Credibility\n- Appeal to Emotion\n- Intentional Fallacy\n- Ad Hominem\n- none\n\nExamples:\n"
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "Senator Randall isn't lying when she says she cares about her constituents—she wouldn't lie to people she cares about."
                    }
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "output_text",
                        "text": "Circular Reasoning"
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "I love eating burgers"
                    }
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "output_text",
                        "text": "none"
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "If we ban Hummers because they are bad for the environment, eventually the government will ban all cars, so we should not ban Hummers."
                    }
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "output_text",
                        "text": "False Dilemma"
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": text_comment
                    }
                ]
            }
        ],
        text={
            "format": {
                "type": "text"
            }
        },
        reasoning={},
        tools=[],
        temperature=0.1,
        max_output_tokens=20,
        top_p=0.8,
        store=True
    )
    return response.output_text


In [13]:
!pip install datasets
from datasets import load_dataset
dataset = load_dataset("tasksource/logical-fallacy")
dataset = dataset.rename_column("source_article", "input")
dataset = dataset.rename_column("logical_fallacies", "label")
test_data = dataset['test']


texts = [sample['input'] for sample in test_data]
labels = [sample['label'] for sample in test_data]
# from datasets import load_dataset
dataset2 = load_dataset("mithrandir22/cocolofa")
dataset2 = dataset2.rename_column("comment", "input")
dataset2 = dataset2.rename_column("fallacy", "label")
test_data2 = dataset2['test']



inputs2 = [sample['input'] for sample in test_data2]
labels2 = [sample['label'] for sample in test_data2]

for i in range(len(inputs2)):
  if labels2[i] == "none":
    texts.append(inputs2[i])
    labels.append(labels2[i])


In [14]:
from tqdm import tqdm
output = []
for text in tqdm(texts, desc="Classifying fallacies"):
    output.append(get_fallacy_response(client,text))

Classifying fallacies:  90%|█████████ | 746/828 [21:59<02:25,  1.77s/it]


KeyboardInterrupt: 

In [16]:
len(output)
testLabels = labels[:746]
len(testLabels)

746

In [17]:
from sklearn.metrics import classification_report
output = [label.lower() for label in output]
# Print classification report with text labels
print(classification_report(testLabels, output))

                                                                                        precision    recall  f1-score   support

                                                                            ad hominem       0.70      0.65      0.67        57
                                                                            ad populum       0.76      0.54      0.63        35
                                                                     appeal to emotion       0.17      0.37      0.23        49
                                                                    circular reasoning       0.92      0.55      0.69        20
                                                                          equivocation       0.25      0.22      0.24         9
                                                                fallacy of credibility       0.29      0.41      0.34        37
                                                                  fallacy of extension       0.29      

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_

In [37]:
!pip install datasets

In [42]:
from datasets import load_dataset

# Load the JSONL file
dataset = load_dataset("json", data_files="/content/labeled_masked_output.jsonl", split="train")

split_dataset = dataset.train_test_split(test_size=0.2, seed=42)

# Access the splits
train_dataset = split_dataset['train']
test_dataset = split_dataset['test']

# Optional: print sizes
print(f"Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")

Train size: 2219, Test size: 555


In [43]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, BertForSequenceClassification
from torch.optim import AdamW
import torch
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import numpy as np
from datasets import load_dataset
import json
import os

# Define the device for training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Load dataset
texts = train_dataset["original"]
labels = train_dataset["label"]

unique_labels = sorted(set(labels))
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}
encoded_labels = [label2id[label] for label in labels]
num_classes = len(unique_labels)

output_dir = "saved_models/m1_model"
os.makedirs(output_dir, exist_ok=True)
with open(f"{output_dir}/label2id.json", "w") as f:
    json.dump(label2id, f)
with open(f"{output_dir}/id2label.json", "w") as f:
    json.dump(id2label, f)

train_texts, val_texts, train_labels, val_labels = train_test_split(texts, encoded_labels, test_size=0.2)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

class FallacyDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_length)
        self.labels = labels

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "label": torch.tensor(self.labels[idx])
        }

    def __len__(self):
        return len(self.labels)

train_dataset = FallacyDataset(train_texts, train_labels, tokenizer)
val_dataset = FallacyDataset(val_texts, val_labels, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_classes)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

criterion = nn.CrossEntropyLoss(weight=class_weights)

num_epochs = 20
patience = 5
best_val_acc = 0
epochs_without_improvement = 4




for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs.logits, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    print(f"Epoch {epoch+1}: Train Loss = {total_loss:.4f}, Train Accuracy = {correct / total:.4f}")

    # Validation loop
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            _, predicted = torch.max(outputs.logits, dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    val_accuracy = correct / total
    print(f"Validation Accuracy = {val_accuracy:.4f}")

    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print("Early stopping triggered")
            break

    scheduler.step()

print(f"Best Validation Accuracy = {best_val_acc:.4f}")

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model and tokenizer saved to {output_dir}")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1: 100%|██████████| 111/111 [00:09<00:00, 11.87it/s]


Epoch 1: Train Loss = 283.5035, Train Accuracy = 0.1290
Validation Accuracy = 0.2500


Epoch 2: 100%|██████████| 111/111 [00:09<00:00, 11.91it/s]


Epoch 2: Train Loss = 238.8486, Train Accuracy = 0.3442
Validation Accuracy = 0.3514


Epoch 3: 100%|██████████| 111/111 [00:09<00:00, 11.91it/s]


...

Epoch 9: Train Loss = 124.1123, Train Accuracy = 0.7583
Validation Accuracy = 0.5158


Epoch 10: 100%|██████████| 111/111 [00:09<00:00, 11.92it/s]


Epoch 10: Train Loss = 123.8018, Train Accuracy = 0.7600
Validation Accuracy = 0.5158


Epoch 11: 100%|██████████| 111/111 [00:09<00:00, 11.94it/s]


Epoch 11: Train Loss = 123.9618, Train Accuracy = 0.7566
Validation Accuracy = 0.5158
Early stopping triggered
Best Validation Accuracy = 0.5248
Model and tokenizer saved to saved_models/m1_model


In [6]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

In [27]:
from datasets import load_dataset

dataset = load_dataset("tasksource/logical-fallacy")

train_dataset = dataset['train']
test_dataset = dataset['test']

print(f"Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")
texts = test_dataset["source_article"]
labels = test_dataset["logical_fallacies"]

Train size: 2680, Test size: 511


In [28]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained("./m1_model")
tokenizer = AutoTokenizer.from_pretrained("./m1_model")
# model_name = 'bert-base-uncased'
# tokenizer = BertTokenizer.from_pretrained(model_name)
# model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(DEVICE)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [29]:
class MyDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len, return_tensors='pt')
        self.labels = labels

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

    def __len__(self):
        return len(self.labels)

test_dataset = MyDataset(texts, labels, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=8)

In [30]:
model.eval()
predictions, true_labels = [], []

unique_labels = list(set(labels))
label_to_index = {label: i for i, label in enumerate(unique_labels)}
numerical_labels = [label_to_index[label] for label in labels]


test_dataset = MyDataset(texts, numerical_labels, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=8)

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        print(logits)
        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())
print(classification_report(true_labels, predictions))



tensor([[-0.1879, -0.8097,  0.9389, -1.1982, -0.0953, -0.6009, -0.5152,  1.9711,
          2.3968, -0.8971, -0.9462, -0.1987,  2.5645, -1.4203],
        [-0.7738, -0.7505, -1.0837,  0.4683, -0.5455, -0.0669,  0.7727, -2.0520,
          2.2052,  0.3112, -0.5769, -0.3461,  2.8458, -2.0849],
        [-1.2701, -1.6086,  0.6791, -0.2447,  0.2904, -0.1015, -0.6116, -0.7891,
          3.2536, -0.6422,  0.8318, -1.1680,  3.0734, -1.4939],
        [-0.6754, -0.7106, -0.6068,  0.1241,  0.6985, -0.5879, -1.1143, -1.2985,
         -0.5979, -0.1141,  5.1670, -1.0002,  0.7398,  0.1770],
        [-1.0203, -1.3166,  1.1442, -0.0090, -0.9811, -0.2019,  0.5229, -1.4718,
          1.0894, -1.4124,  0.3164, -0.3240,  4.7031, -1.9520],
        [-1.2906, -0.7273,  1.1531, -0.9502,  0.3475,  0.1003, -1.3252,  3.5228,
          1.3158, -0.8864, -1.1853,  0.2106,  1.3168, -0.9920],
        [-0.0544,  0.4159,  0.5973, -1.6970,  0.6665, -0.4242, -1.4400,  4.7630,
          0.6173,  0.3850, -1.1474, -0.2798, -0.7

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
